In [1]:
import hoda
import tensorly as tl

print(tl.get_backend())
%pip freeze | grep moabb

cupy
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import MotorImagery
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
tmin = -0.2
tmax=None
fmin = 8
fmax = 30
sfreq = 60

paradigm = MotorImagery(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax,  n_classes=4)
datasets = [
    BNCI2014_001()
]

evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    suffix="hoda_mi",
    overwrite=True,
    random_state=42,
    n_jobs=1,

)

<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
/usr/local/lib/python3.10/dist-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(
Choosing from all possible events


To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.


In [5]:
from sklearn.pipeline import  Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from mne.decoding import Scaler, CSP
import numpy as np
from hoda.hoda import HODA
from hoda.tensorize import STFTensor, Crop
from hoda.classification import Vectorize

pipelines = dict()

component_grid = [1,2,4,8]
def cast_dtype(X):
    return X.astype(np.float64)

#pipelines['CSP+LDA'] = Pipeline([
#    ('scaler', Scaler(scalings='mean')),
#    ('dtype', FunctionTransformer(cast_dtype)),
#    ('clf',  GridSearchCV(
#        Pipeline([
#            ('csp', CSP()),
#            ('lda', LDA())
#        ]),
#        param_grid=dict(csp__n_components=component_grid),
#        n_jobs = len(component_grid)
#    ))
#])

nfreqs=23
pipelines['HODA'] = Pipeline([
        ('scaler', Scaler(scalings='mean')),
        ('tfr', STFTensor(sfreq=sfreq, tmin=tmin,
            morlet_params=dict(freqs=np.linspace(fmin, fmax, nfreqs)),
            baseline_params=dict(baseline=(-.15,-.05), mode='logratio'),
            decim=4
        )),
        ('crop', Crop(begin=0,end=None,sfreq=sfreq, tmin=tmin)),
        ('hoda', HODA(
            max_iter=256,
            tol=1e-8,
            rank=None,
            init='random',
            random_state=42,
            shrinkage='lw',
            toeplitz=None,
            taper=False,
            obj='rt',
            solver='lanczos',
            delta=0.1,
        )),
        ('vec', Vectorize()),
        ('zscore', StandardScaler()),
        ('lda', LDA()),
])

In [6]:
#import warnings
#warnings.filterwarnings("ignore")
import mne
mne.set_log_level(verbose='WARNING')
results = evaluation.process(pipelines)

BNCI2014-001-WithinSession:   0%|                                                                                                       | 0/9 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/loc

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


BNCI2014-001-WithinSession:  11%|██████████▍                                                                                   | 1/9 [04:15<34:06, 255.80s/it]/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/loc

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


BNCI2014-001-WithinSession:  22%|████████████████████▉                                                                         | 2/9 [08:57<31:37, 271.12s/it]/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
/usr/loc

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


BNCI2014-001-WithinSession:  22%|████████████████████▉                                                                         | 2/9 [12:50<44:55, 385.08s/it]

KeyboardInterrupt



In [ ]:
results

In [ ]:
import seaborn as sns
order = results.groupby('pipeline')
order = order.score.aggregate('mean')
order = order.sort_values()

sns.barplot(
     data=results,
    y="score", x="dataset", hue="pipeline", hue_order=order.index,
)

In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
from moabb.analysis.plotting import meta_analysis_plot, paired_plot
_ = meta_analysis_plot(stats, 'tLDA', 'HODA_prune')
_  = paired_plot(results, 'tLDA', 'HODA_prune')